# Learning Objectives

* Build the smallest useful Pegasus workflow — two jobs linked by a file — through the Pegasus Python API.
* Generate the replica, transformation and site catalogs and the workflow DAG from the same class the command-line generator uses.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Read the output to see which nodes the jobs ran on.

# Quickstart (Hello World) Workflow

A two-job pipeline used to learn the Pegasus API and to validate a fresh Pegasus/HTCondor installation before
running a real pipeline. Both jobs run `pegasus-keg.py`, which reads its input file, records the hostname of the
node it ran on, and writes both to its output — so the final output shows where each job executed. See
`README.md` in this repository for the full description.

**Workflow Jobs:**
1. `hello` - reads `f.in`, records the execution host, writes `f.inter`
2. `world` - reads `f.inter`, records the execution host, writes `f.out`

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2 and
Python 3.8+ — all assumed to already be available in this notebook's environment.
[Apptainer](https://apptainer.org/) is only needed for the optional container.

## 1. Prepare Sample Input Data

The only input is a small text file, `input/f.in`. It is in the repository; the generator recreates it with sample
contents if it is missing.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import QuickstartWorkflow, ensure_input_file

INPUT_FILE = ensure_input_file(str(Path("input/f.in").resolve()))
print(open(INPUT_FILE).read())

## 2. Build the Apptainer Container (optional)

The workflow runs on the bare Python interpreter by default. To run the jobs inside a container instead, build
`Apptainer/Quickstart_Container.def` and set `CONTAINER` in the next section to the `.sif` path.

In [ ]:
%%bash
if [ ! -f Apptainer/Quickstart_Container.sif ]; then
    apptainer build Apptainer/Quickstart_Container.sif Apptainer/Quickstart_Container.def
else
    echo "Container already built: Apptainer/Quickstart_Container.sif"
fi

## 3. Generate the Workflow

The following cell uses the `QuickstartWorkflow` class from `workflow_generator.py` — the same generator used by
the command-line tool — to build the Pegasus workflow DAG (site, transformation, and replica catalogs) and writes it
to `workflow.yml`. Nothing about the workflow is defined here: change the pipeline in `workflow_generator.py`, and
both the notebook and the CLI pick it up.

`create_sites_catalog()` writes a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
import argparse

DAGFILE = "workflow.yml"
EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "access-pegasus.yml"; None = local HTCondor catalog below
CONTAINER = None            # e.g. "Apptainer/Quickstart_Container.sif"

# The workflow-specific inputs the CLI would parse
ARGS = argparse.Namespace(spin_time=3)

workflow = QuickstartWorkflow(dagfile=DAGFILE, input_file=INPUT_FILE, container_image=CONTAINER)

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating workflow DAG...")
workflow.create_workflow(ARGS)

workflow.write()
print("\nQuickstart workflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. It shows the two jobs, `hello` then
`world`, linked by the intermediate file `f.inter`.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the `compute` site defined in the site catalog
chosen above.

The CLI version of this generator (`workflow_generator.py`) writes the same workflow non-interactively and
prints the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages its final output to `output/`:

| Output | Description |
|--------|-------------|
| `f.out` | The execution hostnames of both jobs plus the accumulated input contents |

`f.inter` is an intermediate file (`stage_out=False`), so it stays in scratch.

In [ ]:
!ls -ltR output/

`f.out` records the node each job ran on:

In [ ]:
!cat output/f.out